# 課程 08 - 多代理設計模式


## 設定


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 為何使用多代理系統？

現實世界的任務如旅程規劃涉及許多不同種類的專業知識—物流、本地知識、預算等等。由單一代理嘗試處理所有事務很快就會變得難以管理。

多代理系統透過**專業化**來解決這個問題：每個代理專注於一個專業領域，產出比通才更高質素的結果。它們同時提升了**可擴展性**—你可以新增新的代理（例如，航班專家、餐廳評論家）而無需重寫現有的工作流程。這些代理透過結構化的流程鏈接起來，將上下文從一個代理傳遞到下一個。


## 建立專門代理人


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 建立順序工作流程

`WorkflowBuilder` 讓您將代理人串接成有向圖。這裡我們建立一個簡單的兩步驟流程：**TravelPlanner** 擬訂行程，接著由 **TravelConcierge** 審查並增強它。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 添加更多代理到工作流程

多代理模式的一大優勢是擴展非常簡單。以下我們新增了一個 **BudgetReviewer** 代理，它會檢查計劃是否符合旅行者的預算，標記可能超出費用限制的項目，並建議節省開支的替代方案。該工作流程現在依序執行三個代理：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

在本課程中你將學會：

1. **創建專門的代理** — 每個代理都有其專注的角色（規劃、禮賓、預算審核）。
2. **使用 `WorkflowBuilder` 和 `add_edge` 將代理串連成有序工作流程**。
3. **串流輸出多代理管道的結果，追蹤是由哪個代理發言**。
4. **擴展工作流程，在不修改現有代理的情況下加入新代理到鏈中**。

多代理設計模式保持每個代理簡單，同時產生比任何單一代理更豐富且經過更完整審核的結果。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於確保準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議使用專業人工翻譯。本公司對因使用本翻譯內容而產生的任何誤解或誤釋概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
